# VSM Full Benchmark Runner

Notebook này chạy end-to-end cho VSM benchmark:

1. Validate dataset.
2. Chạy inference qua một hoặc nhiều hệ thống.
3. Load `per_case_results.jsonl` và `failures.jsonl`.
4. Tạo bảng tổng hợp.
5. Vẽ visual để đưa vào report.

Mặc định notebook dùng `dry_run` để test pipeline không gọi LLM. Khi chạy thật, đổi `SYSTEMS` trong cell config.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "backend" / "benchmarks" / "vsm" / "data" / "vsm_cases.jsonl").exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError("Cannot find repo root containing backend/benchmarks/vsm/data/vsm_cases.jsonl")

backend_path = repo_root / "backend"
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

repo_root

## Config

Các cấu hình thường dùng:

- Test pipeline nhanh: `SYSTEMS = ["dry_run"]`, `LIMIT = 5`.
- Chạy full dry-run: `SYSTEMS = ["dry_run"]`, `LIMIT = None`.
- Chạy hệ của bạn: `SYSTEMS = ["ours_multi_agent"]`, `SELECTED_MODEL = "slm"` hoặc provider bạn dùng.
- Chạy Kaggle baselines: `SYSTEMS = ["mindchat", "soulchat"]` sau khi set ngrok URL trong `.env`.
- Chạy LLM text baselines: `SYSTEMS = ["base_model", "prompt_1_1"]`, chỉnh `MODEL_TYPE`, `MODEL`.

In [ ]:
from pathlib import Path

DATASET = repo_root / "backend" / "benchmarks" / "vsm" / "data" / "vsm_cases.jsonl"
OUT_DIR = repo_root / "backend" / "benchmarks" / "vsm" / "results" / "notebook_run"

# Safe default: no real LLM call.
SYSTEMS = ["dry_run"]

# Set LIMIT=None for full 190-case run. Keep small while checking setup.
LIMIT = 5
CASE_GROUP = None  # examples: "cbt_dialogues", "mbi_dialogues", "ba_dialogues", "safety_adversarial_cases", "yalom_group_cases"
CASE_ID = None     # comma-separated case ids if needed

# Used by base_model / prompt_1_1.
MODEL_TYPE = "deepseek"
MODEL = "deepseek-chat"

# Used by ours_multi_agent.
SELECTED_MODEL = "slm"

OUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET, OUT_DIR

## Validate Dataset

In [ ]:
from benchmarks.vsm.data.schema import load_vsm_cases, summarize_cases

cases = load_vsm_cases(DATASET)
dataset_summary = summarize_cases(cases)
dataset_summary

In [ ]:
import pandas as pd
from collections import Counter

dataset_tables = {
    "case_groups": pd.DataFrame(dataset_summary["case_groups"].items(), columns=["Case Group", "Cases"]),
    "routes": pd.DataFrame(dataset_summary["routes"].items(), columns=["Route", "Cases"]),
    "difficulties": pd.DataFrame(dataset_summary["difficulties"].items(), columns=["Difficulty", "Cases"]),
    "benchmark_intents": pd.DataFrame(dataset_summary["benchmark_intents"].items(), columns=["Benchmark Intent", "Cases"]),
}

dataset_tables["case_groups"]

In [ ]:
dataset_tables["routes"]

In [ ]:
dataset_tables["difficulties"]

In [ ]:
dataset_tables["benchmark_intents"].sort_values("Cases", ascending=False)

## Optional: Check Remote Baselines

Chỉ cần chạy cell này nếu dùng `mindchat` hoặc `soulchat`.

In [ ]:
from benchmarks.vsm.adapters.remote_chatbots import healthcheck_remote_chatbot

remote_systems = [s for s in SYSTEMS if s in {"mindchat", "soulchat"}]
remote_health = {system: healthcheck_remote_chatbot(system) for system in remote_systems}
remote_health

## Run Inference

Cell này sẽ ghi artifacts vào `OUT_DIR`:

- `per_case_results.jsonl`
- `failures.jsonl`
- `inference_summary.json`

In [ ]:
from benchmarks.vsm.runners.run_inference import run_inference

summary = run_inference(
    dataset=DATASET,
    out_dir=OUT_DIR,
    systems=SYSTEMS,
    limit=LIMIT,
    case_group=CASE_GROUP,
    case_id=CASE_ID,
    model_type=MODEL_TYPE,
    model=MODEL,
    selected_model=SELECTED_MODEL,
)
summary

## Load Results

In [ ]:
import json

per_case_path = OUT_DIR / "per_case_results.jsonl"
failures_path = OUT_DIR / "failures.jsonl"
summary_path = OUT_DIR / "inference_summary.json"

per_case_rows = [json.loads(line) for line in per_case_path.read_text(encoding="utf-8").splitlines() if line.strip()]
failure_rows = [json.loads(line) for line in failures_path.read_text(encoding="utf-8").splitlines() if line.strip()]

len(per_case_rows), len(failure_rows), per_case_path

In [ ]:
turn_rows = []
for case_row in per_case_rows:
    for turn in case_row["turns"]:
        score = turn.get("deterministic_score") or {}
        metadata = turn.get("metadata") or {}
        turn_rows.append({
            "system": case_row["system"],
            "case_id": case_row["case_id"],
            "case_group": case_row["case_group"],
            "route": case_row["route"],
            "risk_level": case_row["risk_level"],
            "difficulty": case_row["difficulty"],
            "benchmark_intent": ",".join(case_row.get("benchmark_intent", [])),
            "turn_id": turn["turn_id"],
            "expected_stage": turn.get("expected_stage"),
            "observed_stage": metadata.get("observed_stage"),
            "expected_peer": turn.get("expected_peer"),
            "observed_peer": metadata.get("observed_peer"),
            "required_technique": turn.get("required_technique"),
            "forbidden_violation": score.get("forbidden_violation"),
            "hard_fail": score.get("hard_fail"),
            "stage_match": score.get("stage_match"),
            "route_match": score.get("route_match"),
            "peer_match": score.get("peer_match"),
            "technique_hint_match": score.get("technique_hint_match"),
            "fallback_used": score.get("fallback_used"),
            "latency_ms": turn.get("latency_ms"),
            "assistant": turn.get("assistant", ""),
        })

turn_df = pd.DataFrame(turn_rows)
failure_df = pd.DataFrame(failure_rows)
turn_df.head()

## Tables

In [ ]:
def pct_true(series):
    non_null = series.dropna()
    if len(non_null) == 0:
        return None
    return round(float(non_null.mean() * 100), 2)

system_table = (
    turn_df.groupby("system")
    .agg(
        turns=("turn_id", "count"),
        cases=("case_id", "nunique"),
        forbidden_pass=("forbidden_violation", lambda s: round(float((~s.fillna(False)).mean() * 100), 2)),
        hard_fail_rate=("hard_fail", lambda s: round(float(s.fillna(False).mean() * 100), 2)),
        stage_accuracy=("stage_match", pct_true),
        route_accuracy=("route_match", pct_true),
        peer_accuracy=("peer_match", pct_true),
        technique_hint_match=("technique_hint_match", pct_true),
        fallback_rate=("fallback_used", lambda s: round(float(s.fillna(False).mean() * 100), 2)),
        avg_latency_ms=("latency_ms", "mean"),
    )
    .reset_index()
)
system_table["avg_latency_ms"] = system_table["avg_latency_ms"].round(2)
system_table

In [ ]:
route_table = (
    turn_df.groupby(["system", "route"])
    .agg(
        turns=("turn_id", "count"),
        cases=("case_id", "nunique"),
        forbidden_pass=("forbidden_violation", lambda s: round(float((~s.fillna(False)).mean() * 100), 2)),
        stage_accuracy=("stage_match", pct_true),
        route_accuracy=("route_match", pct_true),
        peer_accuracy=("peer_match", pct_true),
        fallback_rate=("fallback_used", lambda s: round(float(s.fillna(False).mean() * 100), 2)),
        avg_latency_ms=("latency_ms", "mean"),
    )
    .reset_index()
)
route_table["avg_latency_ms"] = route_table["avg_latency_ms"].round(2)
route_table

In [ ]:
group_table = (
    turn_df.groupby(["system", "case_group"])
    .agg(
        turns=("turn_id", "count"),
        cases=("case_id", "nunique"),
        forbidden_pass=("forbidden_violation", lambda s: round(float((~s.fillna(False)).mean() * 100), 2)),
        hard_fail_rate=("hard_fail", lambda s: round(float(s.fillna(False).mean() * 100), 2)),
        stage_accuracy=("stage_match", pct_true),
        peer_accuracy=("peer_match", pct_true),
        fallback_rate=("fallback_used", lambda s: round(float(s.fillna(False).mean() * 100), 2)),
    )
    .reset_index()
)
group_table

In [ ]:
if failure_df.empty:
    failure_table = pd.DataFrame(columns=["system", "failure_type", "count"])
else:
    failure_table = (
        failure_df.groupby(["system", "failure_type"])
        .size()
        .reset_index(name="count")
        .sort_values(["system", "count"], ascending=[True, False])
    )
failure_table

In [ ]:
safety_table = group_table[group_table["case_group"] == "safety_adversarial_cases"].copy()
yalom_table = group_table[group_table["case_group"] == "yalom_group_cases"].copy()
safety_table, yalom_table

## Save Tables

In [ ]:
tables_dir = OUT_DIR / "tables_from_notebook"
tables_dir.mkdir(parents=True, exist_ok=True)

system_table.to_csv(tables_dir / "system_table.csv", index=False)
route_table.to_csv(tables_dir / "route_table.csv", index=False)
group_table.to_csv(tables_dir / "group_table.csv", index=False)
failure_table.to_csv(tables_dir / "failure_table.csv", index=False)
turn_df.to_csv(tables_dir / "turn_level_results.csv", index=False)

list(tables_dir.glob("*.csv"))

## Visuals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

figures_dir = OUT_DIR / "figures_from_notebook"
figures_dir.mkdir(parents=True, exist_ok=True)

plt.style.use("default")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = system_table.sort_values("forbidden_pass", ascending=False)
ax.bar(plot_df["system"], plot_df["forbidden_pass"], color="#1b9e77")
ax.set_ylim(0, 105)
ax.set_ylabel("Forbidden Pass Rate (%)")
ax.set_title("Overall Deterministic Safety Pass")
ax.tick_params(axis="x", rotation=25)
for i, v in enumerate(plot_df["forbidden_pass"]):
    ax.text(i, v + 1, f"{v:.1f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(figures_dir / "overall_forbidden_pass.png", dpi=180)
plt.show()

In [ ]:
metric = "forbidden_pass"
pivot = route_table.pivot(index="route", columns="system", values=metric).fillna(0)
fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind="bar", ax=ax)
ax.set_ylim(0, 105)
ax.set_ylabel("Forbidden Pass Rate (%)")
ax.set_title("Route-Level Safety Pass")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(figures_dir / "route_forbidden_pass.png", dpi=180)
plt.show()

In [ ]:
if not failure_table.empty:
    pivot = failure_table.pivot(index="failure_type", columns="system", values="count").fillna(0)
    fig, ax = plt.subplots(figsize=(10, 6))
    pivot.plot(kind="bar", stacked=True, ax=ax)
    ax.set_ylabel("Failure Count")
    ax.set_title("Failure Taxonomy")
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    fig.savefig(figures_dir / "failure_taxonomy.png", dpi=180)
    plt.show()
else:
    print("No failures recorded.")

In [ ]:
pivot = group_table.pivot(index="case_group", columns="system", values="forbidden_pass").fillna(0)
fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(kind="bar", ax=ax)
ax.set_ylim(0, 105)
ax.set_ylabel("Forbidden Pass Rate (%)")
ax.set_title("Case-Group Performance")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(figures_dir / "case_group_forbidden_pass.png", dpi=180)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
latency_plot = system_table.sort_values("avg_latency_ms")
ax.bar(latency_plot["system"], latency_plot["avg_latency_ms"], color="#377eb8")
ax.set_ylabel("Average Latency (ms)")
ax.set_title("Runtime Latency")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(figures_dir / "avg_latency.png", dpi=180)
plt.show()

In [ ]:
if "peer_accuracy" in group_table.columns:
    peer_df = group_table[group_table["case_group"].isin(["yalom_group_cases", "safety_adversarial_cases", "mbi_dialogues"])]
    if not peer_df.empty and peer_df["peer_accuracy"].notna().any():
        pivot = peer_df.pivot(index="case_group", columns="system", values="peer_accuracy").fillna(0)
        fig, ax = plt.subplots(figsize=(10, 5))
        pivot.plot(kind="bar", ax=ax)
        ax.set_ylim(0, 105)
        ax.set_ylabel("Peer Accuracy (%)")
        ax.set_title("Peer Policy Accuracy")
        ax.tick_params(axis="x", rotation=25)
        fig.tight_layout()
        fig.savefig(figures_dir / "peer_policy_accuracy.png", dpi=180)
        plt.show()
    else:
        print("No peer accuracy metadata available for current systems.")

## Inspect Failures / Examples

In [ ]:
if failure_df.empty:
    print("No deterministic failures.")
else:
    display(failure_df.head(20))

In [ ]:
# Random-ish sample of model outputs for manual audit.
sample_cols = ["system", "case_id", "case_group", "route", "turn_id", "assistant", "forbidden_violation", "stage_match", "peer_match"]
turn_df[sample_cols].head(10)

## Next Step

Sau khi notebook chạy xong với các hệ thật, dùng các file trong `OUT_DIR` để build report chính thức hoặc làm LLM judge/human audit subset.